In [ ]:
import numpy as np
from scipy.special import iv, kv
from scipy.optimize import root_scalar
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

def L(lam, n_c, R0):
    sqrtlamR0 = np.sqrt(lam) * R0
    sqrtlamncR0 = np.sqrt(lam * n_c) * R0
    numerator = iv(0, sqrtlamncR0) * iv(1, sqrtlamR0) - np.sqrt(n_c) * iv(1, sqrtlamncR0) * iv(0, sqrtlamR0)
    denominator = iv(0, sqrtlamncR0) * kv(1, sqrtlamR0) + np.sqrt(n_c) * iv(1, sqrtlamncR0) * kv(0, sqrtlamR0)
    return numerator / denominator

def compute_b1(b0, lam, n_c, R0):
    return b0 * L(lam, n_c, R0)

def compute_b0(lam, n_c, c_B, R1, R0):
    sqrtlamR1 = np.sqrt(lam) * R1
    term1 = iv(0, sqrtlamR1)
    term2 = L(lam, n_c, R0) * kv(0, sqrtlamR1)
    return c_B / (term1 + term2)

def compute_A(b0, b1, lam, R0, c_bar):
    sqrtlamR0 = np.sqrt(lam) * R0
    return (R0 / np.sqrt(lam)) * (b0 * iv(1, sqrtlamR0) - b1 * kv(1, sqrtlamR0)) - (R0**2 / 2) * c_bar

def compute_B(b0, b1, lam, R0, c_bar, A):
    sqrtlamR0 = np.sqrt(lam) * R0
    return (1 / lam) * (b0 * iv(0, sqrtlamR0) + b1 * kv(0, sqrtlamR0)) - (R0**2 / 4) * c_bar - A * np.log(R0)

def final_equation(R0, lam, n_c, c_bar, c_B, R1):
    b0 = compute_b0(lam, n_c, c_B, R1, R0)
    b1 = compute_b1(b0, lam, n_c, R0)
    A = compute_A(b0, b1, lam, R0, c_bar)
    B = compute_B(b0, b1, lam, R0, c_bar, A)
    sqrtlamR1 = np.sqrt(lam) * R1
    term1 = (R1**2 / 4) * c_bar
    term2 = A * np.log(R1)
    term3 = B
    term4 = - (1 / lam) * (b0 * iv(0, sqrtlamR1) + b1 * kv(0, sqrtlamR1))
    return term1 + term2 + term3 + term4

def R1_rhs(t, R1, lam, n_c, c_bar, c_B, model_G0):
    sol = root_scalar(final_equation, args=(lam, n_c, c_bar, c_B, R1), bracket=[0.0001, 0.99 * R1], method='brentq')
    if not sol.converged:
        raise RuntimeError(f"Failed to find R0 at t={t}, R1={R1}")
    R0 = sol.root

    b0 = compute_b0(lam, n_c, c_B, R1, R0)
    b1 = compute_b1(b0, lam, n_c, R0)
    A = compute_A(b0, b1, lam, R0, c_bar)

    sqrtlamR1 = np.sqrt(lam) * R1
    term1 = (1 / lam) * (b0 * iv(1, sqrtlamR1) - b1 * kv(1, sqrtlamR1))
    term2 = -(R1 / 2) * c_bar
    term3 = - A / R1

    return (term1 + term2 + term3) * model_G0

In [ ]:
lam = 1;
c_B = 10;
c_bar = 0.5 * c_B
n_c = 1.e-3;
R1_0 = 2.5
model_G0 = 1.

In [ ]:
from scipy.interpolate import splprep, splev

def compute_equiv_radius(bdry):
    # 确保曲线闭合
    if not np.allclose(bdry[0], bdry[-1]):
        bdry = np.vstack([bdry, bdry[0]])

    # 构造参数样条（周期条件 periodic=True）
    tck, u = splprep([bdry[:, 0], bdry[:, 1]], s=0, per=True)

    # 细化参数用于积分
    u_fine = np.linspace(0, 1, 2000)
    dx_du, dy_du = splev(u_fine, tck, der=1)

    # 积分得到曲线长度
    length = np.trapezoid(np.sqrt(dx_du**2 + dy_du**2), u_fine)

    # 等效半径
    return length / (2 * np.pi)

def plot_radii_over_time_with_ode(steps_to_plot, bdry_points_0_list, bdry_points_1_list, numerical_delta_t, t_ode_span, t_ode_steps):
    """
    Plot numerical boundary radii and ODE solution curves on the same figure.
    """
    t_ode_eval = np.linspace(t_ode_span[0], t_ode_span[1], t_ode_steps)
    sol = solve_ivp(
        fun=lambda t, R1: R1_rhs(t, R1[0], lam, n_c, c_bar, c_B, model_G0),
        t_span=t_ode_span,
        y0=[R1_0],
        t_eval=t_ode_eval,
        method='RK45'
    )

    t_ode = sol.t
    R1_ode = sol.y[0]
    
    R0_ode = []
    for R1 in R1_ode:
        sol_R0 = root_scalar(final_equation, args=(lam, n_c, c_bar, c_B, R1), bracket=[0.0001, 0.99 * R1], method='brentq')
        if not sol_R0.converged:
            raise RuntimeError(f"R0 not found for R1 = {R1}")
        R0_ode.append(sol_R0.root)
    R0_ode = np.array(R0_ode)


    radii_0 = []
    radii_1 = []

    for t in steps_to_plot:
        bdry0 = bdry_points_0_list[t]
        bdry1 = bdry_points_1_list[t]

        r0 = compute_equiv_radius(bdry0)
        r1 = compute_equiv_radius(bdry1)

        radii_0.append(r0)
        radii_1.append(r1)

    # Determine x-axis for numerical data
    x_vals = [t * numerical_delta_t for t in steps_to_plot]
    x_label = f"Time"
    
    # Plotting
    plt.figure(dpi=600)

    # Numerical results
    plt.plot(x_vals, radii_0, label='Numerical R0(t)', color='blue', linestyle='-', linewidth = 1)
    plt.plot(x_vals, radii_1, label='Numerical R1(t)', color='black', linestyle='-', linewidth = 1)

    # ODE solution curves
    plt.plot(t_ode, R0_ode, label='Reference R0(t)', color='red', linestyle='-', linewidth = 2, alpha = 0.5)
    plt.plot(t_ode, R1_ode, label='Reference R1(t)', color='green', linestyle='-', linewidth = 2, alpha = 0.5)

    plt.xlabel(x_label)
    plt.ylabel("Radius")
    plt.title("Numerical vs Reference Radii Over Time")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    R0_ode = np.array(R0_ode)[::round(numerical_delta_t * 10000)]
    radii_0 = np.array(radii_0)
    error0 = R0_ode - radii_0
    R1_ode = np.array(R1_ode)[::round(numerical_delta_t * 10000)]
    radii_1 = np.array(radii_1)
    error1 = R1_ode - radii_1

    return np.linalg.norm(error0, 2) / np.sqrt(len(error0) - 1), np.linalg.norm(error1, 2) / np.sqrt(len(error1) - 1)

In [ ]:
import os
import numpy as np
import re 

def extract_number(filename):
    match = re.search(r'_(\d+)\.txt$', filename)
    return int(match.group(1)) if match else -1

data_dict = {
    "ctrl_points_0": [],  # bdry_points_*.txt
    "ctrl_points_1": [],  # bdry_points_*.txt
}

base_dirs = ["solutions", "points"]

for base_dir in base_dirs:
    if not os.path.exists(base_dir):
        continue
    
    for filename in sorted(os.listdir(base_dir), key=extract_number):  
        filepath = os.path.join(base_dir, filename)
        
        if filename.startswith("ctrl_points_0_"):
            data_dict["ctrl_points_0"].append(np.loadtxt(filepath))
        elif filename.startswith("ctrl_points_1_"):
            data_dict["ctrl_points_1"].append(np.loadtxt(filepath))
        
ctrl_points_0_list = [np.array(arr) for arr in data_dict["ctrl_points_0"]]
ctrl_points_1_list = [np.array(arr) for arr in data_dict["ctrl_points_1"]]

In [ ]:
numerical_delta_t = 0.01
t_end = 0.5

In [ ]:
assert(len(ctrl_points_0_list) == len(ctrl_points_1_list))

plot_radii_over_time_with_ode(
    steps_to_plot = np.arange(len(ctrl_points_0_list)),
    bdry_points_0_list = ctrl_points_0_list,
    bdry_points_1_list = ctrl_points_1_list,
    numerical_delta_t = numerical_delta_t,
    t_ode_span = (0.0, t_end), 
    t_ode_steps = int(t_end / 0.0001) + 1
)